## Set Up

In [1]:
# strategies/opening_reversal.py

import pandas as pd

In [2]:
import yfinance as yf
import pandas as pd
from config import TICKERS, INTERVAL, LOOKBACK_DAYS, GAP_THRESHOLD, HOLDING_PERIOD_HOURS
from strategies.opening_reversal import calculate_signals
from backtest.backtest_runner import backtest
import matplotlib.pyplot as plt


## Analysis

In [ ]:
df['date'] = df.index.date
df = df.reset_index(drop=True)
df.columns = [col[0] for col in df.columns]


# Merge signals
df = df.merge(signals[['date', 'signal']], on='date', how='left')
df['signal'].fillna(0, inplace=True)

returns = []
position = 0
entry_price = None
exit_time = None

for idx, row in df.iterrows():
current_time = row['time']

# Hard stop at 16:00 every day
if current_time >= time(16, 0) and position != 0:
    exit_price = row['Close']
    if position == 1:
        returns.append((exit_price - entry_price) / entry_price)
    else:
        returns.append((entry_price - exit_price) / entry_price)
    position = 0
    entry_price = None
    exit_time = None

if position == 0:
    if row['signal'] == 1:
        position = 1
        entry_price = row['Close']
        entry_datetime = datetime.combine(datetime.today(), current_time)
        exit_datetime = entry_datetime + timedelta(hours=holding_period_hours)
        exit_time = min(exit_datetime.time(), time(16, 0))  # cap at 4PM
    elif row['signal'] == -1:
        position = -1
        entry_price = row['Close']
        entry_datetime = datetime.combine(datetime.today(), current_time)
        exit_datetime = entry_datetime + timedelta(hours=holding_period_hours)
        exit_time = min(exit_datetime.time(), time(16, 0))
else:
    if current_time >= exit_time:
        exit_price = row['Close']
        if position == 1:
            returns.append((exit_price - entry_price) / entry_price)
        else:
            returns.append((entry_price - exit_price) / entry_price)
        position = 0
        entry_price = None
        exit_time = None
